---
title: "Ionosphere module"
bibliography: ../bibliography.bib
---


# Table of Contents

1. [Ionospheric delay in GNSS observations](#ionospheric-delay-in-gnss-observations)
    - [Definition of STEC](#definition-of-stec)
    - [Ionospheric delay in code and phase](#ionospheric-delay-in-code-and-phase)
    - [Observation equations](#observation-equations)
    - [Geometry-free combination](#geometry-free-combination)
    - [Biases and STEC observables](#biases-and-stec-observables)
    - [Phase-to-code leveling](#phase-to-code-leveling)
    - [Mapping STEC to VTEC](#mapping-stec-to-vtec)
2. [Current GNX-py ionosphere workflow](#current-gnx-py-ionosphere-workflow)
3. [Notebook setup](#notebook-setup)
4. [Example of STEC processing in GNX-py](#example-of-stec-processing-in-gnx-py)
5. [Inspecting STEC arcs](#inspecting-stec-arcs)
6. [Estimating VTEC above the station](#estimating-vtec-above-the-station)
7. [Comparing measured STEC with models](#comparing-measured-stec-with-models)
8. [Bias policy diagnostics](#bias-policy-diagnostics)
9. [BeiDou B1IB3I STEC smoke example](#beidou-b1ib3i-stec-smoke-example)
10. [What to try next](#what-to-try-next)


# Ionospheric delay in GNSS observations

Ionospheric delay is one of the key factors degrading GNSS positioning. Long-established methods of dealing with it include modelling ionospheric delay or using a combination of observations at two frequencies. The ionosphere-free combination removes the first-order ionospheric delay, but it does so at the cost of increased observation noise [@Handbook; @HofmannWellenhof2008]. Ionospheric modelling and monitoring is a broad field of GNSS-related research (e.g. [@Mannucci1998; @Schaer1999]).

GNX-py offers functionality for measuring, modelling and monitoring the spatial and temporal changes of the ionosphere. In this notebook we will focus on the introductory workflow: measuring Slant TEC (STEC) for one station, comparing it with ionosphere models, and understanding the diagnostic columns that the current library returns.

**Current version note.** The current `gnx_py.ionosphere` workflow supports GPS, Galileo and BeiDou through the shared signal registry. `TECSession` processes one constellation at a time, so for a multi-GNSS comparison we run separate sessions for `G`, `E` and `C`. For BDS, `B1IB3I` is the best-tested STEC pair in the repository at the moment.

**Notation**

- $f_i$: carrier frequency $i$ [Hz],
- $\lambda_i = c/f_i$: wavelength,
- $c$: speed of light,
- $N_e(s)$: electron density [el/m$^3$],
- $\mathrm{STEC} = \int_{\mathrm{LOS}} N_e(s)\,ds$ [el/m$^2$],
- $1~\mathrm{TECU} = 10^{16}~\mathrm{el/m^2}$,
- $\rho$: geometric range,
- $T$: tropospheric delay,
- $\delta t_r, \delta t_s$: receiver/satellite clock offsets,
- $P_i$: code measurement on $f_i$ [m],
- $\Phi_i = \lambda_i \varphi_i$: carrier phase observation [m],
- $d^{P}_{(\cdot),i}, d^{L}_{(\cdot),i}$: hardware delays,
- $N_i$: carrier ambiguity.


## Definition of STEC

Ionospheric delay on the satellite-receiver path is defined as the line integral of electron density along the signal path:

$$
\mathrm{STEC} = \int_{\mathrm{LOS}} N_e(s)\,ds \qquad [\mathrm{el/m^2}].
$$

The commonly accepted unit for describing STEC is TECU, Total Electron Content Unit:

$$
1~\mathrm{TECU} = 10^{16}~\mathrm{el/m^2}.
$$

STEC is a slant quantity. It depends on the satellite-receiver geometry. When the satellite is low above the horizon, the signal travels a longer path through the ionosphere and STEC increases even if the vertical ionosphere above the station is unchanged.


## Ionospheric delay in code and phase

In GNSS processing we usually want the ionospheric delay in metres for a given signal. For first-order ionospheric delay:

$$
I_i = \frac{40.3\cdot 10^{16}}{f_i^2}\,\mathrm{STEC}.
$$

Here $I_i$ is the ionospheric delay in metres for signal $i$, while STEC is expressed in TECU. The sign is different for code and phase observations: the ionosphere delays the group velocity of code observations and advances the carrier phase.

$$
P_i: +I_i \quad\text{(group delay)}, \qquad
\Phi_i: -I_i \quad\text{(phase advance)}.
$$

This sign difference is the reason why dual-frequency code and phase observations are so useful for ionospheric work. The ionosphere is dispersive, while many other terms in the observation equation are not.


## Observation equations

Recording code and phase observations as $P_i$ and $\Phi_i$:

$$
\begin{aligned}
P_{i} &= \rho + c(\delta t_r-\delta t_s) + T + I_i
+ d^{P}_{r,i}-d^{P}_{s,i} + \varepsilon_{P_i}, \\[2mm]
\Phi_{i} &= \rho + c(\delta t_r-\delta t_s) + T - I_i
+ \lambda_i N_i + d^{L}_{r,i}-d^{L}_{s,i} + \varepsilon_{\Phi_i}.
\end{aligned}
$$

The terms $\rho$, receiver/satellite clock and troposphere are non-dispersive. They appear in the same way on both frequencies. The first-order ionosphere and hardware delays are frequency-dependent, which is why a two-frequency combination can expose them.


## Geometry-free combination

STEC measurement using GNSS is possible thanks to the geometry-free combination. It removes all non-dispersive terms, such as the geometric range, clocks and tropospheric delay. For two frequencies:

$$
\begin{aligned}
P_{\mathrm{GF}} &= P_1 - P_2 = (I_1 - I_2)
+ \Big[(d^P_{r,1}-d^P_{r,2})-(d^P_{s,1}-d^P_{s,2})\Big] + \eta_P,\\[1mm]
\Phi_{\mathrm{GF}} &= \Phi_1 - \Phi_2 = -(I_1 - I_2)
+ (\lambda_1N_1-\lambda_2N_2)
+ \Big[(d^L_{r,1}-d^L_{r,2})-(d^L_{s,1}-d^L_{s,2})\Big] + \eta_\Phi.
\end{aligned}
$$

The code geometry-free observable is noisy but absolute after bias correction. The phase geometry-free observable is much cleaner, but it contains an unknown constant ambiguity term for each continuous satellite arc. GNX-py therefore detects cycle slips and tracking gaps, splits observations into arcs, and levels the phase combination to the code combination.


## Biases and STEC observables

The remaining biases in the geometry-free combination are differences in hardware delays between two frequencies for satellites and receivers. Satellite DCB/DSB/OSB values can be introduced from external products. Receiver DCB values can be taken from GIM/IONEX auxiliary blocks, manually configured, estimated in a calibration workflow, or set to zero when we only need a diagnostic run.

GNX-py currently uses the geometry-free code convention

```text
P4 = P2 - P1
```

inside the STEC bias policy. OSB values are differenced as $OSB(P_1)-OSB(P_2)$, while DCB/DSB products are treated as differential pair biases for the selected code pair. The source order is controlled by `TECConfig.stec_bias_sources`, typically:

```python
("product", "gim", "config", "zero")
```

After applying the selected bias convention, code-derived STEC can be written as

$$
\mathrm{STEC}^{(P)} =
\frac{P_{\mathrm{GF}} - \mathrm{DCB}^{P}_{12}}
{40.3\left(\tfrac{1}{f_1^2}-\tfrac{1}{f_2^2}\right)}.
$$

For phase observations, with constant $B_\Phi$:

$$
\mathrm{STEC}^{(\Phi)} =
-\frac{\Phi_{\mathrm{GF}} - B_\Phi}
{40.3\left(\tfrac{1}{f_1^2}-\tfrac{1}{f_2^2}\right)}.
$$

**Current version note.** The output dataframe now includes explicit bias diagnostics such as `stec_sat_bias_m`, `stec_station_bias_m`, `stec_total_bias_m`, `stec_sat_bias_source`, `stec_station_bias_source`, `stec_bias_code_1` and `stec_bias_code_2`. These columns are part of the scientific interpretation. If you change the signal pair or the product source, inspect them before interpreting the STEC values.


## Phase-to-code leveling

$B_{\Phi}$ is a linear combination of phase ambiguities at both frequencies. Because of this bias, the phase geometry-free observation is precise but not on the correct absolute scale. To remove it, we use phase-to-code leveling: the phase observation is shifted to match the code observation over a local window [@Mannucci1998].

One simple form is

$$
\tilde{L}_4^{(n)} = L_4^{(n)} + \frac{1}{K} \sum_{k=1}^{K} \Bigl[P_4^{(k)} - L_4^{(k)}\Bigr].
$$

**where:**

- $\tilde{L}_4^{(n)}$ is the levelled carrier phase geometry-free observation at epoch $n$,
- $P_4^{(k)}$ is the code geometry-free observation at epoch $k$,
- $L_4^{(k)}$ is the phase geometry-free observation at epoch $k$,
- $K$ is the number of samples used for smoothing/leveling.

In GNX-py, the corresponding configuration parameters are `leveling_ws` and `median_leveling_ws`. The output keeps several versions of the measured TEC, for example `code_tec`, `phase_tec`, `leveled_tec`, `median_leveled_tec`, `leveled_tec_sg` and `leveled_tec_cheby`. The point is not that one column is always best. The point is that the dataframe exposes enough diagnostic information to see how the measurement was built.


## Mapping STEC to VTEC

In ionosphere modelling using GNSS, it is usually assumed that electrons are concentrated in an infinitely thin shell at a certain altitude, for example 350, 400 or 450 km. The conversion between STEC and VTEC is based on this geometry. Assume shell height $h_{\mathrm{ion}}$, Earth radius $R_E$ and satellite elevation $E$:

$$
M(E) = \frac{1}{\sqrt{1-\left(\tfrac{R_E}{R_E+h_{\mathrm{ion}}}\cos E\right)^2}}.
$$

Then

$$
\mathrm{STEC} \approx M(E)\cdot \mathrm{VTEC}, \qquad
\mathrm{VTEC} \approx \frac{\mathrm{STEC}}{M(E)}.
$$

This mapping function is simple, but it is good enough for an introductory tutorial. More advanced mapping functions and shell assumptions can be found in the literature and are important when building operational ionosphere products.


# Current GNX-py ionosphere workflow

The introductory STEC pipeline in GNX-py is controlled by `TECConfig` and executed by `TECSession`:

1. `TECConfig` selects the station file, constellation, signal pair, orbit source, ionosphere model and bias policy.
2. `TECSession` loads observations and products, interpolates satellite geometry, applies selected corrections and detects cycle slips.
3. `STECMonitor` groups observations by satellite arc, applies the configured bias policy, performs phase-to-code leveling, and returns a dataframe with both STEC values and diagnostics.

Important practical points in the current version:

- `TECSession` processes one constellation at a time: use `use_sys="G"`, `"E"` or `"C"`.
- GPS and Galileo are the historical paths; BeiDou is active, with `B1IB3I` currently the best-tested BDS pair.
- `compare_models` can add `klobuchar`, `ntcm` and/or `gim` comparison columns, but this path should be treated as model-comparison support, not as a certification of every product combination.
- Bias source selection is explicit through `stec_bias_sources` and `stec_missing_bias`.
- Advanced activity indexes and kriging exist in the module, but they are intentionally outside this introduction.


# Notebook setup

The notebook uses the repository `sample_data/` directory and avoids local absolute paths. The main station in this tutorial is `BRUX`, because the current sample data contains a compact observation file for that station together with navigation, SP3, ANTEX, DCB/OSB and GIM products.


In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import contextlib
import importlib.util
import io
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import gnx_py as gnx
from gnx_py.ionosphere import TECConfig, TECSession


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "gnx_py").is_dir():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from inside the cloned GNX-py repository, "
        "or set REPO_ROOT manually to the repository directory."
    )


REPO_ROOT = find_repo_root(Path.cwd())
SAMPLE_DATA = REPO_ROOT / "sample_data"
TUTORIAL_DIR = REPO_ROOT / "tutorial" / "ionosphere"

print(f"gnx_py imported from: {gnx.__file__}")
print(f"sample data: {SAMPLE_DATA}")


In [ ]:
def load_tutorial_module(module_name: str, module_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError(f"Cannot load {module_name} from {module_path}")
    spec.loader.exec_module(module)
    return module


tools = load_tutorial_module("ionosphere_tutorial_tools", TUTORIAL_DIR / "tools.py")

df_head = tools.df_head
plot_stec = tools.plot_stec
VTECZenithEstimator = tools.VTECZenithEstimator
STECModelComparator = tools.STECModelComparator


In [ ]:
STATION = "BRUX"
DOY = 35

OBS_PATH = SAMPLE_DATA / "BRUX00BEL_R_20240350000_01D_30S_MO.crx.gz"
NAV_PATH = SAMPLE_DATA / "BRDC00IGS_R_20240350000_01D_MN.rnx"
SP3_PATHS = [
    SAMPLE_DATA / "GRG0MGXFIN_20240340000_01D_05M_ORB.SP3",
    SAMPLE_DATA / "GRG0MGXFIN_20240350000_01D_05M_ORB.SP3",
    SAMPLE_DATA / "GRG0MGXFIN_20240360000_01D_05M_ORB.SP3",
]
ATX_PATH = SAMPLE_DATA / "igs20.atx"
DCB_PATH = SAMPLE_DATA / "CAS1OPSRAP_20240350000_01D_01D_DCB.BIA"
GIM_PATH = SAMPLE_DATA / "COD0OPSFIN_20240350000_01D_01H_GIM.INX"

GPS_TLIM = [datetime(2024, 2, 4, 0, 0), datetime(2024, 2, 4, 0, 30)]
BDS_TLIM = [datetime(2024, 2, 4, 0, 0), datetime(2024, 2, 4, 0, 20)]

required_paths = [OBS_PATH, NAV_PATH, ATX_PATH, DCB_PATH, GIM_PATH, *SP3_PATHS]
missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing sample files:\n" + "\n".join(str(path) for path in missing))


# Example of STEC processing in GNX-py

Let's now move on to an example of STEC measurement in GNX-py. The procedure is:

- obtain satellite coordinates at the time of signal transmission,
- introduce corrections that matter for the geometry-free combination, especially PCO and code biases,
- detect cycle slips and loss of tracking,
- split observations into continuous satellite arcs,
- level phase observations to code observations,
- convert metres/electrons into TECU and inspect the result.

In the dataframe you will see arc identifiers such as `G02_0_1`. I read them as: satellite `G02`, tracking-gap arc `0`, cycle-slip arc `1`. This is why one physical satellite can appear as several arc-labelled entries.

For simplicity, the whole pipeline is enclosed in `TECSession`, managed by `TECConfig`. We will start with GPS because it is the clearest introductory case, and then run a very short BeiDou `B1IB3I` example to show how the current bias policy generalizes to BDS.


## TECConfig for this tutorial

Below we build a small helper around `TECConfig`. The configuration is intentionally explicit, because STEC values depend on processing choices:

| parameter | why it matters here |
|---|---|
| `sys` / `use_sys` | `TECSession` processes one constellation at a time |
| `gps_freq`, `gal_freq`, `bds_freq` | select the two signals used in the geometry-free combination |
| `orbit_type` | `broadcast` uses NAV interpolation; `precise` uses SP3 interpolation |
| `ionosphere_model` | adds GIM-based model columns and can provide GIM DCB data |
| `compare_models` | requests extra model columns, e.g. `klobuchar` and `ntcm` |
| `add_dcb`, `add_sta_dcb` | enable satellite and receiver/station bias corrections |
| `stec_bias_sources` | defines fallback order: product, GIM, config, zero |
| `stec_missing_bias` | choose whether missing bias should warn/use zero or raise |
| `leveling_ws`, `median_leveling_ws` | control phase-to-code leveling windows |
| `time_limit` | keeps tutorial runs light |

The tutorial uses `orbit_type="broadcast"` to keep the examples fast. The same `TECConfig` can be adapted to precise SP3 geometry by changing `orbit_type` and supplying `sp3_path`.


In [ ]:
def run_tec(config: TECConfig, use_sys: str):
    """Run TECSession while keeping notebook output compact."""
    stream = io.StringIO()
    with warnings.catch_warnings(record=True) as captured, contextlib.redirect_stdout(stream):
        warnings.simplefilter("always")
        result = TECSession(config=config, use_sys=use_sys).run()
    warning_messages = list(dict.fromkeys(str(item.message) for item in captured))
    return result, warning_messages


def build_tec_config(
    *,
    system: str = "G",
    time_limit=GPS_TLIM,
    compare_models=("ntcm", "klobuchar"),
    ev_mask: int = 30,
) -> TECConfig:
    return TECConfig(
        obs_path=OBS_PATH,
        nav_path=NAV_PATH,
        sp3_path=SP3_PATHS,
        atx_path=ATX_PATH,
        dcb_path=DCB_PATH,
        gim_path=GIM_PATH,
        sys=system,
        gps_freq="L1L2",
        gal_freq="E1E5a",
        bds_freq="B1IB3I",
        orbit_type="broadcast",
        ionosphere_model="gim",
        troposphere_model=False,
        time_limit=time_limit,
        day_of_year=DOY,
        station_name=STATION,
        screen=False,
        ev_mask=ev_mask,
        use_gfz=True,
        windup=True,
        rel_path=True,
        sat_pco=False,
        rec_pco=True,
        add_dcb=True,
        add_sta_dcb=True,
        rcv_dcb_source="gim",
        stec_bias_sources=("product", "gim", "config", "zero"),
        stec_missing_bias="warn_zero",
        compare_models=list(compare_models) if compare_models else False,
        use_iono_rms=True,
        leveling_ws=20,
        median_leveling_ws=20,
        min_arc_len=2,
    )


TEC_ELECTRON_COLUMNS = [
    "leveled_tec",
    "median_leveled_tec",
    "code_tec",
    "phase_tec",
    "leveled_tec_sg",
    "leveled_tec_cheby",
]
GPS_L1_HZ = 1575.42e6
L1_TECU_TO_M = 40.3e16 / GPS_L1_HZ**2


def with_tecu_units(df: pd.DataFrame) -> pd.DataFrame:
    """Return a display copy with STEC columns in TECU and model delays in TECU."""
    out = df.copy()
    for column in TEC_ELECTRON_COLUMNS:
        if column in out.columns:
            out[column] = out[column] / 1e16
    for column in ["ion", "klobuchar", "ntcm"]:
        if column in out.columns:
            out[f"{column}_tecu"] = out[column] / L1_TECU_TO_M
    return out


def compact_columns(df: pd.DataFrame, columns: list[str]) -> list[str]:
    return [column for column in columns if column in df.columns]


## Running GPS STEC

We will use the BRUX observation file from `sample_data`. The run is limited to 30 minutes so that the notebook remains light. For a real analysis you would extend `GPS_TLIM` to a longer interval and then store summary products outside the notebook.


In [ ]:
gps_config = build_tec_config(system="G", time_limit=GPS_TLIM, compare_models=("ntcm", "klobuchar"))
obs_tec, gps_warnings = run_tec(gps_config, use_sys="G")

print(f"rows={len(obs_tec)}")
print(f"satellite arcs={obs_tec.index.get_level_values('sv').nunique()}")
print(f"unique captured warnings={len(gps_warnings)}")


The resulting dataframe contains both the original observation/preprocessing columns and the ionosphere-specific outputs. The columns `ion`, `klobuchar` and `ntcm` are modelled delays in metres on L1. The measured STEC columns are initially in electrons per square metre, so in the next step we create a display copy in TECU.


In [ ]:
raw_preview_cols = compact_columns(
    obs_tec,
    [
        "ion", "klobuchar", "ntcm", "leveled_tec", "code_tec", "phase_tec",
        "ev", "az", "stec_sat_bias_source", "stec_station_bias_source",
    ],
)
df_head(obs_tec[raw_preview_cols], nrows=5, ncols=10, floatfmt=".3f", truncate_str=14)


In [ ]:
obs_tec_tecu = with_tecu_units(obs_tec)

preview_cols = compact_columns(
    obs_tec_tecu,
    [
        "leveled_tec", "median_leveled_tec", "code_tec", "phase_tec",
        "ion_tecu", "klobuchar_tecu", "ntcm_tecu", "ev",
    ],
)
df_head(obs_tec_tecu[preview_cols], nrows=5, ncols=8, floatfmt=".2f", truncate_str=14)


The contrast between `code_tec` and `phase_tec` is the practical version of the theory above. Code STEC is noisy but absolute after bias correction. Phase STEC is precise but shifted by ambiguity. `leveled_tec` is the phase-derived observable placed on the code scale.


# Inspecting STEC arcs

Before plotting a STEC series, it is useful to look at the arcs that survived the preprocessing. A long continuous arc is easier to interpret than a very short arc near a cycle slip.


In [ ]:
arc_counts = obs_tec_tecu.groupby(level="sv").size().sort_values()
selected_arc = arc_counts.idxmax()

fig, ax = plt.subplots(figsize=(8, 7))
arc_counts.plot(kind="barh", ax=ax, grid=True)
ax.set_xlabel("Number of observations")
ax.set_title("Observation count per satellite arc")
plt.tight_layout()

print(f"Selected arc for the STEC plot: {selected_arc}")


The helper `plot_stec` shows the same arc through several lenses: code STEC, phase STEC, levelled STEC and smoothed variants. This is not just a pretty plot. It is a sanity check that cycle slip detection and leveling produced a physically plausible arc.


In [ ]:
plot_stec(obs_tec_tecu, selected_arc, time_col="LT")


# Estimating VTEC above the station

With STEC measured, corrected for satellite and receiver biases, and levelled with phase observations, we can estimate a rough VTEC series above the station. The helper `VTECZenithEstimator` uses the thin-shell mapping function and selects the highest-elevation observations at each epoch. Then it interpolates and smooths the resulting series.

This is intentionally presented as a teaching tool, not as the most rigorous VTEC estimator. It helps connect the geometry from the first half of the notebook with a concrete time series above one receiver.


In [ ]:
estimator = VTECZenithEstimator(
    df=obs_tec_tecu,
    time_col="LT",
    elev_col="ev",
    stec_col="leveled_tec",
    sv_col="sv",
    min_elev_deg=30,
    resample_freq="5min",
    smooth_method="whittaker",
    smooth_params={"lam": 50, "order": 2},
)

df_vtec, zen = estimator.run()

plot_frame = obs_tec_tecu.copy()
plot_frame["sat"] = plot_frame.index.get_level_values("sv").str[:3]

fig, ax = plt.subplots(figsize=(8, 5))
for sv, group in plot_frame.groupby("sat"):
    group = group.reset_index()
    ax.scatter(group["LT"], group["leveled_tec"], label=sv, s=2)

ax.plot(zen["LT"], zen["vtec_smooth"], color="black", linewidth=2, label="Smoothed VTEC")
ax.set_title(f"STEC measured at station {STATION} and rough zenith VTEC")
ax.set_ylabel("TEC [TECU]")
ax.set_xlabel("Local time")
ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(loc="best", ncol=2, fontsize=7)
plt.tight_layout()


In [ ]:
df_head(zen, nrows=5, ncols=7, floatfmt=".2f", truncate_str=14)


In the `zen` dataframe, `vtec` is the mapped value at the selected high-elevation link, while `vtec_smooth` is the smoothed zenith series. The result should be interpreted gently: for a single station and a short time interval this is more of an intuition-building exercise than a final ionosphere product.


# Comparing measured STEC with models

At the beginning we asked `TECConfig` to compute GIM, Klobuchar and NTCM comparison columns. Now we can compare those modelled values with measured `leveled_tec`. The model columns were returned in metres of L1 delay, so we use the TECU-converted display columns created earlier: `ion_tecu`, `klobuchar_tecu` and `ntcm_tecu`.


In [ ]:
model_cols = compact_columns(obs_tec_tecu, ["ion_tecu", "klobuchar_tecu", "ntcm_tecu"])
model_names = [
    {"ion_tecu": "GIM", "klobuchar_tecu": "Klobuchar", "ntcm_tecu": "NTCM G"}[column]
    for column in model_cols
]

comparator = STECModelComparator(
    df=obs_tec_tecu,
    x_col="leveled_tec",
    model_cols=model_cols,
    model_names=model_names,
    scale=1.0,
    gridsize=50,
    mincnt=1,
)

stats_table = comparator.report(
    title="Modelled vs measured STEC",
    show_hex=True,
    show_hist=True,
    return_stats=True,
)


In [ ]:
stats_table = stats_table.set_index("Model")
df_head(stats_table, ncols=10, floatfmt=".2f", reset_index=True)


The point of this comparison is not to declare a global winner from one station and half an hour of data. It is to learn how to read the diagnostics. A good model should stay near the line $y=x$, have a small bias, and show a relatively compact error histogram. A GIM product will often agree well when its own DCB information participates in the processing chain, so the bias-source diagnostics are part of the interpretation.


# Bias policy diagnostics

STEC can look convincing even when the wrong bias convention was used. For this reason the current GNX-py output includes bias-source columns. Let us summarize them for the GPS run.


In [ ]:
bias_diag_cols = compact_columns(
    obs_tec_tecu,
    [
        "stec_sat_bias_source", "stec_station_bias_source",
        "stec_bias_code_1", "stec_bias_code_2",
        "stec_sat_bias_m", "stec_station_bias_m", "stec_total_bias_m",
    ],
)

bias_summary = (
    obs_tec_tecu.reset_index()
    .groupby([col for col in ["stec_sat_bias_source", "stec_station_bias_source", "stec_bias_code_1", "stec_bias_code_2"] if col in obs_tec_tecu.columns])
    .size()
    .reset_index(name="rows")
)

df_head(bias_summary, nrows=10, ncols=8, floatfmt=".3f", reset_index=False)


The summary tells us which source was actually used. `product_dsb` means that the DCB/DSB product provided a matching differential bias for the selected code pair. If you see `gim`, `config` or `zero`, that does not automatically mean the run is wrong; it means the interpretation must include that fallback.


# BeiDou B1IB3I STEC smoke example

BeiDou is supported in the current ionosphere workflow. The safest default pair for the sample data is `B1IB3I`, which maps to the legacy B1I/B3I observable pair. This mirrors the caution from the SIS tutorial: modern BDS signals can be configured when the observations and bias products support them, but they require validation before scientific use.

The short BDS run below is intentionally compact. Its purpose is to show that the same `TECConfig` pattern works for `system="C"`, and that the output carries the same STEC and bias diagnostics.


In [ ]:
bds_config = build_tec_config(
    system="C",
    time_limit=BDS_TLIM,
    compare_models=False,
    ev_mask=20,
)
bds_tec, bds_warnings = run_tec(bds_config, use_sys="C")
bds_tec_tecu = with_tecu_units(bds_tec)

print(f"rows={len(bds_tec)}")
print(f"BDS satellite arcs={bds_tec.index.get_level_values('sv').nunique()}")
print(f"unique captured warnings={len(bds_warnings)}")

df_head(
    bds_tec_tecu[compact_columns(
        bds_tec_tecu,
        [
            "leveled_tec", "code_tec", "phase_tec", "ev",
            "stec_sat_bias_source", "stec_station_bias_source",
            "stec_bias_code_1", "stec_bias_code_2",
        ],
    )],
    nrows=5,
    ncols=10,
    floatfmt=".2f",
    truncate_str=14,
)


In [ ]:
bds_bias_summary = (
    bds_tec_tecu.reset_index()
    .groupby([col for col in ["stec_sat_bias_source", "stec_station_bias_source", "stec_bias_code_1", "stec_bias_code_2"] if col in bds_tec_tecu.columns])
    .size()
    .reset_index(name="rows")
)

df_head(bds_bias_summary, nrows=10, ncols=8, floatfmt=".3f", reset_index=False)


For BDS `B1IB3I`, the diagnostic code pair should correspond to the legacy BDS observables used by the current tests. If you change `bds_freq` to a modern pair such as `B1C`, `B2a` or `B2b`, first check whether the observation file and bias product contain the matching code observables. During validation runs, `stec_missing_bias="raise"` is safer than quietly falling back to zero.


# What to try next

This concludes the updated introductory ionosphere tutorial. You have seen the theory behind STEC, the geometry-free combination, bias handling, phase-to-code leveling, a rough VTEC estimate, model comparison, and a short BeiDou `B1IB3I` run.

For your own experiments, I recommend changing only one thing at a time:

- extend `GPS_TLIM` and inspect whether the same arcs remain stable,
- switch `system` between `G`, `E` and `C`, remembering that `TECSession` processes one constellation at a time,
- test `stec_missing_bias="raise"` to catch missing bias products early,
- compare `leveled_tec`, `median_leveled_tec`, `leveled_tec_sg` and `code_tec`,
- move to the calibration notebook when you want to estimate receiver DCB instead of using GIM/product values.

The advanced notebooks on activity indexes and kriging remain separate, data-dependent topics. They should be updated later when the network data are available.
